In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
import time
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import os

In [ ]:
# Constants
RandomSTATE = 1082
MODEL_NAME = "StackingClassifier(LogisticRegression RandomForest SVC)"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [ ]:
EXP = protocol.real_heldout("2d")
feature_names = EXP.feature_names
X_train_scaled = EXP.X_train
X_test_scaled = EXP.X_test
y_train = EXP.y_train
y_test = EXP.y_test


In [5]:
print("Number of classes:", len(set(y)))
print("Class distribution:\n", y.value_counts())

Number of classes: 4
Class distribution:
 Output
0    397825
2     44232
1     36458
3     32015
Name: count, dtype: int64


In [6]:
# Define base models
base_learners = [
    ('lr', LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga')),
    ('rf', RandomForestClassifier(random_state=RandomSTATE)),
    ('svc', SVC(probability=True, random_state=RandomSTATE))
]

# Meta model
meta_learner = LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga')

In [7]:
# Create and train the Stacking model
stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    passthrough=False,
    n_jobs=-1
)
start_time = time.time()
stacking_model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time

In [8]:
# Predict and evaluate
y_pred = stacking_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Stacked Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Stacked Model Accuracy on Test Set: 92.88%
F1 Score: 0.9272
Confusion Matrix:
[[309413   2283   5592    972]
 [  3425  23081   2660      0]
 [  9439   3029  22918      0]
 [  1678      3      0  23931]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    318260
           1       0.81      0.79      0.80     29166
           2       0.74      0.65      0.69     35386
           3       0.96      0.93      0.95     25612

    accuracy                           0.93    408424
   macro avg       0.87      0.84      0.85    408424
weighted avg       0.93      0.93      0.93    408424



In [ ]:
from eval_metrics import evaluate, summary, time_inference
from gps_ids.results import append_validated

_proba = stacking_model.predict_proba(X_test_scaled) if hasattr(stacking_model, "predict_proba") else None
_timing = time_inference(stacking_model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    family="Stacking",
    y_proba=_proba,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=stacking_model.get_params(),
    timing=_timing,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
